# Text versus Embedding: Incremental Geopolitical-Exposure Discovery

This notebook tests whether holdings-based asset embeddings identify event-sensitive firms whose own pre-event filings did not produce a direct exposure label. It uses only frozen artifacts: **no WRDS query, OpenAI call, return-informed label construction, or embedding refit occurs here**.

The primary chain is direct pre-event text → leave-self-out embedding inference → event-return validation. Active/passive investor subsets appear only as a final robustness reference. Event returns validate market relevance, not the underlying business mechanism; leading discoveries still require independent economic evidence.

In [ ]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT=Path.cwd()
if ROOT.name!='geopolitical_exposure': ROOT=ROOT/'geopolitical_exposure'
sys.path.insert(0,str(ROOT))
OUTPUT=ROOT/'incremental_discovery'; FIGURES=OUTPUT/'figures'
OUTPUT.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)
from src.incremental_discovery import CAR_COLUMNS, build_panel, run_model_comparison, evaluate_embedding_only, rank_candidates
sns.set_theme(style='whitegrid',context='notebook')
print('Output directory:',OUTPUT)

## 1. Audit immutable inputs

The required checkpoints are hashed for provenance. The analysis fails early if any frozen input is missing.

In [ ]:
required=[ROOT/'text_expansion_v2/processed/expanded_300_firms_frozen.parquet',ROOT/'text_expansion_v2/results/expanded_response_audit.parquet',ROOT/'embedding_expansion_v2/directional_firm_channel_seeds.parquet',ROOT/'embedding_expansion_v2/embedding_propagated_scores.parquet',ROOT/'embedding_expansion_v2/leave_one_seed_out_predictions.parquet',ROOT/'embedding_expansion_v2/characteristic_propagated_scores.parquet',ROOT/'event_data/market_model_cars.parquet']
audit=[]
for path in required:
    if not path.exists(): raise FileNotFoundError(path)
    audit.append({'path':str(path.relative_to(ROOT)),'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
input_audit=pd.DataFrame(audit); input_audit.to_parquet(OUTPUT/'input_audit.parquet',index=False)
display(input_audit[['path','bytes']])

## 2. Separate text states and construct leave-self-out scores

`direct_exposure` requires accepted evidence from the firm's own filing. `reviewed_no_channel_exposure` means valid text was reviewed but no exposure was accepted for that channel. `text_unavailable` is never coded as zero. For seed firms, the ordinary propagated value is replaced by a prediction produced after removing that firm's label.

In [ ]:
panel,seeds=build_panel(ROOT,representation='all_s12')
panel.to_parquet(OUTPUT/'text_embedding_event_panel.parquet',index=False)
state_summary=panel.groupby('text_state').agg(firm_channel_rows=('firm_id','size'),firms=('firm_id','nunique')).reset_index()
coverage=pd.DataFrame({'metric':['corpus_firms','text_reviewed_firms','direct_seed_firms','embedding_only_candidate_firms','firms_with_event_returns'],'value':[panel.firm_id.nunique(),panel.loc[panel.text_reviewed,'firm_id'].nunique(),panel.loc[panel.text_state.eq('direct_exposure'),'firm_id'].nunique(),panel.loc[panel.embedding_only_candidate,'firm_id'].nunique(),panel.loc[panel.car_0_1.notna(),'firm_id'].nunique()]})
display(state_summary); display(coverage)

## 3. Compare text, embeddings, characteristics, and the combined model

The common sample contains only firms with reviewed text and observed event returns. Scores are standardized within channel. Every regression controls for log market equity and SIC2 fixed effects and uses HC3 standard errors. The key comparison is whether text plus embeddings improves on text alone.

In [ ]:
models=run_model_comparison(panel)
models.to_parquet(OUTPUT/'incremental_model_comparison.parquet',index=False)
display(models.sort_values(['outcome','channel','adjusted_r2'],ascending=[True,True,False]))

## 4. Test embedding-only discoveries

This test excludes every direct firm-channel text label. Within reviewed but unlabeled firms, it measures rank correlation, the top-minus-bottom embedding-score quintile spread, and directional agreement. A positive spread means returns moved with the embedding-implied direction.

In [ ]:
discovery=evaluate_embedding_only(panel)
discovery.to_parquet(OUTPUT/'embedding_only_event_validation.parquet',index=False)
display(discovery)

## 5. Produce a return-blind discovery list

Candidates are ranked only by the magnitude of their standardized embedding score; returns do not enter selection. The strongest cases should be checked against pre-event customer, supplier, geographic-revenue, government-contract, segment, and commodity evidence.

In [ ]:
ranked=rank_candidates(panel,count=20)
ranked.to_parquet(OUTPUT/'top_embedding_only_discoveries.parquet',index=False)
display(ranked)

## 6. Visualize the incremental test

The left panel compares information sets. The right panel focuses exclusively on firms missed by direct channel-level text classification. These are discovery diagnostics rather than causal estimates.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(16,6))
m=models.query("outcome=='car_0_5'")
sns.barplot(data=m,x='channel',y='adjusted_r2',hue='specification',ax=axes[0]); axes[0].set_title('Information-set comparison: CAR[0,5]')
d=discovery.query("outcome=='car_0_5'")
sns.barplot(data=d,x='channel',y='top_minus_bottom_car',color='#4472C4',ax=axes[1]); axes[1].axhline(0,color='black',lw=1); axes[1].set_title('Embedding-only firms: top minus bottom CAR[0,5]')
for ax in axes:
    ax.set_xlabel(''); ax.tick_params(axis='x',rotation=30)
    for container in ax.containers:
        try: ax.bar_label(container,fmt='%.3f',padding=2,fontsize=8)
        except Exception: pass
plt.tight_layout(); plt.savefig(FIGURES/'text_vs_embedding_incremental_discovery.png',dpi=200,bbox_inches='tight'); plt.show()

## 7. Investor-source robustness—not the primary question

The active/passive representations answer only whether the result depends on the investor population used to learn the asset geometry. They do not define the geopolitical experiment.

In [ ]:
robustness=pd.read_parquet(ROOT/'event_study_v2/locked_model_comparison.parquet')[['representation','channel','outcome','firms','spearman_rho','spearman_bh_qvalue','top_minus_bottom_car','incremental_adjusted_r2']]
robustness.to_parquet(OUTPUT/'investor_source_robustness_reference.parquet',index=False)
display(robustness.query("outcome=='car_0_5'").sort_values(['channel','spearman_rho'],ascending=[True,False]))

## 8. Interpretation boundary

Incremental discovery requires convergence of evidence: a leave-self-out embedding relationship, event returns in the predicted direction, improvement over direct text and characteristic similarity, placebo resistance, and an independently documented mechanism for leading firms. A return association establishes event sensitivity—not a supply-chain or operational mechanism. Failure to detect exposure in a filing may reflect absent or indirect disclosure rather than an LLM error. The defensible claim is complementarity: direct text identifies explicit exposure, while embeddings may extend it to economically related firms.

In [ ]:
inventory=[]
for path in sorted(OUTPUT.glob('*.parquet')):
    inventory.append({'artifact':path.name,'rows':len(pd.read_parquet(path)),'bytes':path.stat().st_size,'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
inventory=pd.DataFrame(inventory)
manifest={'stage':'text_vs_embedding_incremental_discovery','primary_representation':'all_s12','channels':sorted(seeds.channel.unique()),'car_windows':CAR_COLUMNS,'artifacts':inventory.to_dict('records')}
(OUTPUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
display(inventory)
print('Complete. Review candidate mechanisms independently before making discovery claims.')